In [1]:
import os

In [2]:
%pwd

'd:\\pr_fol\\data_science_prj\\chicken_project\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\pr_fol\\data_science_prj\\chicken_project'

In [6]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class preparecallbacksConfig:
    root_dir: Path
    tensorboard_root_log_dir: Path
    checkpoint_model_filepath: Path

In [9]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

In [16]:
class configurationManger:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])


    def get_prepare_callbacks_config(self) -> preparecallbacksConfig:
        config = self.config.prepare_callbacks
        model_ckpt_dir = Path(config.checkpoint_model_filepath)
        create_directories([
            Path(model_ckpt_dir),
            Path(config.tensorboard_root_log_dir)

        ])

        prepare_callbacks_config = preparecallbacksConfig(
            root_dir=Path(config.root_dir),
            tensorboard_root_log_dir=Path(config.tensorboard_root_log_dir),
            checkpoint_model_filepath=Path(config.checkpoint_model_filepath)
        )

        return prepare_callbacks_config



In [17]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
import time

In [18]:
class preparecallback:
    def __init__(self, config: preparecallbacksConfig):
        self.config = config


    @property
    def _create_tb_callback(self):
        timestamp = time.strftime("%Y-%m-%d-%H-%M-%S")
        tb_running_log_dir = os.path.join(
        self.config.tensorboard_root_log_dir, 
        f'tb_logs_at_{timestamp}'
        )

        return tf.keras.callbacks.TensorBoard(log_dir=tb_running_log_dir)    


    @property
    def _create_ckpt_callback(self):
        return tf.keras.callbacks.ModelCheckpoint(
            filepath=self.config.checkpoint_model_filepath,
            save_best_only=True
        )


    @property
    def get_tb_ckpt_callbacks(self):
        return [
            self._create_tb_callback,
            self._create_ckpt_callback
        ]




In [19]:
try:
    config = configurationManger()
    prepare_callbacks_config = config.get_prepare_callbacks_config()
    prepare_callbacks = preparecallback(config=prepare_callbacks_config)
    callback_list = prepare_callbacks.get_tb_ckpt_callbacks()


except Exception as e:
    raise e

[2026-10-14 16:05:45,631: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-10-14 16:05:45,644: INFO: common: yaml file: params.yaml loaded successfully]
[2026-10-14 16:05:45,654: INFO: common: created directory at: artifacts]
[2026-10-14 16:05:45,666: INFO: common: created directory at: artifacts\prepare_callbacks\checkpoint_dir\model.h5]
[2026-10-14 16:05:45,674: INFO: common: created directory at: artifacts\prepare_callbacks\tensorboard_log_dir]


TypeError: 'list' object is not callable